In [1]:
import torch

x = torch.ones(5)  # input tensor
y = torch.zeros(3)  # expected output
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
z = torch.matmul(x, w)+b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

print(z.grad_fn)
grad_fn=loss.grad_fn
print(f"z.grad_fn.next_functions: {z.grad_fn.next_functions}")

print(f"是否是 Function 实例？ {isinstance(loss.grad_fn, torch.autograd.Function)}")

# 方法3: 检查是否有 backward 方法（Function 的标志）
print("=== 方法检查 ===")
print(f"hasattr(grad_fn, 'requires_grad'): {hasattr(grad_fn, 'requires_grad')}")
print(f"hasattr(grad_fn, 'next_functions'): {hasattr(grad_fn, 'next_functions')}")
print()


# 验证：loss.grad_fn 就是 BinaryCrossEntropyWithLogitsBackward0
print(f"loss.grad_fn: {loss.grad_fn}")
print(f"类型: {type(loss.grad_fn)}")
print(f"类型名: {type(loss.grad_fn).__name__}")

print("=== 调用 backward 前 ===")
print(f"x.grad: {w.grad}")

loss.backward()

print("=== 调用 backward 后 ===")
print(f"x.grad: {w.grad}")

print(dir(grad_fn))

z.grad_fn.next_functions: ((<SqueezeBackward4 object at 0x11bdc7df0>, 0), (<AccumulateGrad object at 0x1052f3a90>, 0))
是否是 Function 实例？ False
=== 方法检查 ===
hasattr(grad_fn, 'requires_grad'): True
hasattr(grad_fn, 'next_functions'): True

loss.grad_fn: <BinaryCrossEntropyWithLogitsBackward0 object at 0x11babf9a0>
类型: <class 'BinaryCrossEntropyWithLogitsBackward0'>
类型名: BinaryCrossEntropyWithLogitsBackward0
=== 调用 backward 前 ===
x.grad: None
=== 调用 backward 后 ===
x.grad: tensor([[0.1955, 0.3219, 0.0494],
        [0.1955, 0.3219, 0.0494],
        [0.1955, 0.3219, 0.0494],
        [0.1955, 0.3219, 0.0494],
        [0.1955, 0.3219, 0.0494]])
['__call__', '__class__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '_input_metadata', '_raw

- 计算图的算子可以自定义 当然可以支持矩阵运算 甚至是复杂函数 比如交叉熵
- 广播不一定体现在计算图里，而是数学上的广播。本质是复合函数的链式求导法则

In [48]:
from torch.autograd import Function


class Power(Function):
    @staticmethod
    def forward(ctx, i):
        print("forward",i)
        result = i**2
        ctx.save_for_backward(i)
        return result
    @staticmethod
    def backward(ctx, grad_output):
        print("backward grad_output",grad_output)
        result, = ctx.saved_tensors
        print("backward result",result)
        grad_result= grad_output * result*2
        print("backward grad_result",grad_result)
        return grad_result
# Use it by calling the apply method:
input=torch.tensor(10,requires_grad=True,dtype=torch.float)
output = Power.apply(input)
print("Power.apply 结果的梯度函数",output)

loss = torch.nn.functional.binary_cross_entropy_with_logits(output, input)
print(loss.grad_fn.next_functions)
loss.backward()

# print(output.grad_fn.next_functions)
# print(f"是否是 Function 实例？ {isinstance(output.grad_fn, Exp)}"1

forward tensor(10., requires_grad=True)
Power.apply 结果的梯度函数 tensor(100., grad_fn=<PowerBackward>)
((<torch.autograd.function.PowerBackward object at 0x119906050>, 0), (<AccumulateGrad object at 0x119593dc0>, 0))
backward grad_output tensor(-9.)
backward result tensor(10., requires_grad=True)
backward grad_result tensor(-180.)
